# Flickr30k T1→S1 Dense Mapped Logit-KD — seed 123

ExpansionNet v2 có vocabulary 10.000 token; Student Flickr30k có vocabulary
7.456 token. Notebook tạo mapping riêng cho Flickr30k, ánh xạ phân phối
Teacher dày đặc bằng `scatter_add`, chuẩn hóa lại khối xác suất và tối ưu:

`L = 0,5 CE_human + 0,5 × T² × KL(q_mapped || p_student)`, `T = 4`.

## Kaggle input bắt buộc — đúng 7 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_S1_123`.
4. Output `F30K_T1_Pseudo_Full`.
5. Dataset `Flickr Image dataset`.
6. Kaggle Model `expansionnet_v2_ieee_bigdata2023`.
7. Package/dataset `pycocoevalcap-1.2` offline.

Khi resume, gắn thêm đúng một output gần nhất của
`F30K_T1S1_DenseMap_123`. Không gắn mapping MS COCO hoặc output KD seed khác.

Runtime: GPU T4, Internet Off. Physical/accum/effective batch `1/64/64`;
tối đa 20 epoch; Early Stopping từ epoch 5, patience 4. Notebook lưu
`last.pt` mỗi 100 optimizer-step. Test luôn `SEALED / NOT USED`.

## Cell 1 — Authorization, integrity và Test-seal

In [1]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, subprocess, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

MODEL_ID = "S1"
MODEL_NAME = "MobileNetV3-Large"
SEED = 123
EXPERIMENT_ID = "F30K_T1S1_DenseMap_123"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
PROTOCOL_PATH = unique_named("F30K_BASELINE_PROTOCOL.json")
TRAIN_PATH = unique_named("f30k_train.parquet")
VAL_PATH = unique_named("f30k_validation.parquet")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")

base_ready = json.loads(BASE_READY_PATH.read_text(encoding="utf-8"))
data_contract = json.loads(DATA_CONTRACT_PATH.read_text(encoding="utf-8"))
protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
vocab = json.loads(VOCAB_PATH.read_text(encoding="utf-8"))

assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert (
    SEED == int(protocol["development_seed"])
    or SEED in [int(x) for x in protocol["confirmation_seeds"]]
)
assert protocol["vocabulary_size"] == 7456 == len(vocab["tokens"])
assert protocol["maximum_epochs"] == 20
assert protocol["selection_metric"] == "Validation CIDEr-D"
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False

for file_name, expected in {
    "f30k_train.parquet": data_contract["artifacts"]["train"]["sha256"],
    "f30k_validation.parquet": data_contract["artifacts"]["validation"]["sha256"],
    "f30k_vocabulary_minfreq5.json": data_contract["artifacts"]["vocabulary"]["sha256"],
}.items():
    actual_path = {"f30k_train.parquet": TRAIN_PATH,
                   "f30k_validation.parquet": VAL_PATH,
                   "f30k_vocabulary_minfreq5.json": VOCAB_PATH}[file_name]
    assert sha256(actual_path) == expected, file_name

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    if wheels:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
    else:
        packages = [p for p in INPUT_ROOT.rglob("pycocoevalcap") if p.is_dir()]
        assert packages, "Hãy gắn input pycocoevalcap-1.2 offline."
        sys.path.insert(0, str(packages[0].parent))

assert importlib.util.find_spec("pycocoevalcap") is not None
assert shutil.which("java"), "METEOR chính thức yêu cầu Java runtime."
assert torch.cuda.is_available(), "Notebook này yêu cầu GPU T4."
DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("Preflight             : PASSED")
print("Experiment            :", EXPERIMENT_ID)
print("Model                 :", MODEL_NAME)
print("Train/Validation      : VERIFIED")
print("Vocabulary            :", len(vocab["tokens"]))
print("Selection             : Validation CIDEr-D")
print("Test                   : SEALED / NOT READ")

Preflight             : PASSED
Experiment            : F30K_T1S1_DenseMap_123
Model                 : MobileNetV3-Large
Train/Validation      : VERIFIED
Vocabulary            : 7456
Selection             : Validation CIDEr-D
Test                   : SEALED / NOT READ


## Cell 2 — S1 architecture và Validation pipeline

In [2]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
assert train_df["image_id"].nunique() == 29000 and len(train_df) == 145000
assert val_df["image_id"].nunique() == 1014 and len(val_df) == 5070
assert set(train_df.image_id).isdisjoint(set(val_df.image_id))

needed_names = {f"{int(x)}.jpg" for x in pd.concat([train_df.image_id, val_df.image_id])}
image_candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in needed_names]
IMAGE_BY_NAME = {}
duplicate_alias_count = 0
for path in image_candidates:
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
        continue
    if old.resolve() == path.resolve():
        continue
    # Một số bản đóng gói Kaggle lưu cùng ảnh ở cả
    # flickr30k_images/<file> và flickr30k_images/flickr30k_images/<file>.
    # Chấp nhận alias lồng nhau khi kích thước byte giống hệt và luôn chọn
    # đường dẫn nông/ngắn nhất. Nếu kích thước khác nhau, dừng để tránh ánh xạ
    # nhầm hai ảnh chỉ tình cờ có cùng tên.
    assert old.stat().st_size == path.stat().st_size, (
        "Hai ảnh cùng tên nhưng khác kích thước", path.name, old, path
    )
    duplicate_alias_count += 1
    IMAGE_BY_NAME[path.name] = min(
        [old, path],
        key=lambda p: (len(p.parts), len(str(p)), str(p)),
    )
missing = sorted(needed_names - set(IMAGE_BY_NAME))
assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])

train_groups = []
for image_id, group in train_df.groupby("image_id", sort=True):
    train_groups.append((int(image_id), [list(map(int, x)) for x in group.token_ids]))
val_groups = []
VAL_REFERENCES = {}
for image_id, group in val_df.groupby("image_id", sort=True):
    iid = int(image_id)
    val_groups.append((iid, [list(map(int, x)) for x in group.token_ids]))
    VAL_REFERENCES[iid] = [str(x) for x in group.caption.tolist()]

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CaptionDataset(Dataset):
    def __init__(self, rows, transform, training):
        self.rows, self.transform, self.training, self.epoch = rows, transform, training, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        image_id, captions = self.rows[index]
        caption = captions[(self.epoch + image_id % len(captions)) % len(captions)] if self.training else captions[0]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as im:
            image = self.transform(im.convert("RGB"))
        return image_id, image, torch.tensor(caption, dtype=torch.long)

def collate(batch):
    image_ids, images, captions = zip(*batch)
    max_len = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), max_len, dtype=torch.long)
    mask = torch.zeros(len(captions), max_len, dtype=torch.bool)
    for i, x in enumerate(captions):
        tokens[i, :len(x)] = x
        mask[i, :len(x)] = True
    return torch.tensor(image_ids), torch.stack(images), tokens, mask

train_set = CaptionDataset(train_groups, train_transform, True)
val_set = CaptionDataset(val_groups, val_transform, False)

def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    g = torch.Generator().manual_seed(SEED + epoch)
    return DataLoader(train_set, batch_size=32, shuffle=True, generator=g, num_workers=2,
                      pin_memory=True, persistent_workers=False, collate_fn=collate)

val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=2,
                        pin_memory=True, persistent_workers=True, collate_fn=collate)

def select_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, dict):
        return [v for v in x.values() if torch.is_tensor(v)][-1]
    if isinstance(x, (tuple, list)):
        return [v for v in x if torch.is_tensor(v)][-1]
    raise TypeError(type(x))

class VisualEncoder(nn.Module):
    def __init__(self, backbone, source_dim):
        super().__init__()
        self.backbone = backbone
        self.source_dim = source_dim
        self.projection = nn.Linear(source_dim, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)
        nn.init.normal_(self.visual_position_embedding, std=0.02)
    def forward(self, images):
        if MODEL_ID == "S1":
            feat = self.backbone(images)
        else:
            feat = select_tensor(self.backbone.forward_features(images))
        assert feat.ndim == 4 and feat.shape[1] == self.source_dim and tuple(feat.shape[-2:]) == (7, 7), feat.shape
        tokens = feat.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(self.projection(tokens) + self.visual_position_embedding)

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.d_model, self.max_steps = 256, 31
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(31, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(256, 8, 1024, 0.1, "gelu",
                                           batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))
        nn.init.normal_(self.token_embedding.weight, std=256 ** -0.5)
        nn.init.normal_(self.position_embedding.weight, std=0.02)
        with torch.no_grad():
            self.token_embedding.weight[0].zero_()
    def forward(self, token_ids, memory, attention_mask=None):
        n, t = token_ids.shape
        pos = torch.arange(t, device=token_ids.device).unsqueeze(0).expand(n, -1)
        x = self.embedding_dropout(self.token_embedding(token_ids) * 16.0 + self.position_embedding(pos))
        causal = torch.triu(torch.ones(t, t, dtype=torch.bool, device=token_ids.device), diagonal=1)
        key_pad = None if attention_mask is None else ~attention_mask.bool()
        x = self.transformer_decoder(x, memory, tgt_mask=causal, tgt_key_padding_mask=key_pad)
        return F.linear(self.final_norm(x), self.token_embedding.weight, self.output_bias)

if MODEL_ID == "S1":
    from torchvision.models import mobilenet_v3_large
    base = mobilenet_v3_large(weights=None).features
elif MODEL_ID == "S2":
    import timm
    base = timm.create_model("mobilevit_s.cvnets_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
elif MODEL_ID == "S3":
    import timm
    base = timm.create_model("tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
else:
    raise ValueError(MODEL_ID)

encoder = VisualEncoder(base, 960)

# Decoder Flickr30k được khởi tạo mới bằng SEED hiện tại; không nạp decoder
# MS COCO có vocabulary 10.080.
torch.manual_seed(SEED)
decoder = CaptionDecoder(len(vocab["tokens"]))
encoder.to(DEVICE)
decoder.to(DEVICE)

for module in encoder.modules():
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())
assert enc_params == 3231024, (enc_params, 3231024)
assert dec_params == 5_084_960, dec_params
print("Images train/val       :", len(train_groups), "/", len(val_groups))
print("Nested image aliases   :", duplicate_alias_count, "(deduplicated)")
print("Encoder/decoder params :", enc_params, "/", dec_params)
print("Test                   : SEALED / NOT READ")

Images train/val       : 29000 / 1014
Nested image aliases   : 30014 (deduplicated)
Encoder/decoder params : 3231024 / 5084960
Test                   : SEALED / NOT READ


## Cell 3 — ExpansionNet, Flickr30k dense mapping và paired data

In [3]:
import gc
import pickle
import re
from argparse import Namespace
from torchvision.transforms import functional as TF

TEACHER_ID = "T1"
TEACHER_NAME = "ExpansionNet v2"
BASELINE_EXPERIMENT_ID = f"F30K_S1_{SEED}"
BASELINE_RESULT_PATH = unique_named(BASELINE_EXPERIMENT_ID + "_RESULT.json")
PSEUDO_RESULT_PATH = unique_named("F30K_T1_Pseudo_Full_RESULT.json")
PSEUDO_PARQUET_PATH = unique_named("pseudocaptions.parquet")
baseline_result = json.loads(BASELINE_RESULT_PATH.read_text(encoding="utf-8"))
pseudo_result = json.loads(PSEUDO_RESULT_PATH.read_text(encoding="utf-8"))

assert baseline_result["status"] == "complete"
assert baseline_result["experiment_id"] == BASELINE_EXPERIMENT_ID
assert baseline_result["model_id"] == "S1"
assert int(baseline_result["seed"]) == SEED
assert baseline_result["test_set_status"] == "sealed"
assert baseline_result["test_set_used"] is False
assert pseudo_result["status"] == "complete"
assert pseudo_result["teacher_id"] == "T1"
assert pseudo_result["records"] == pseudo_result["unique_image_ids"] == 29000
assert pseudo_result["test_set_status"] == "sealed"
assert pseudo_result["test_set_used"] is False
assert sha256(PSEUDO_PARQUET_PATH) == pseudo_result["parquet"]["sha256"]

BASELINE_BEST_PATH = BASELINE_RESULT_PATH.parent / "best.pt"
assert sha256(BASELINE_BEST_PATH) == baseline_result["artifacts"]["best.pt"]["sha256"]
baseline_payload = torch.load(BASELINE_BEST_PATH, map_location="cpu", weights_only=False)
assert baseline_payload["experiment_id"] == BASELINE_EXPERIMENT_ID
encoder.load_state_dict(baseline_payload["encoder_state_dict"], strict=True)
decoder.load_state_dict(baseline_payload["decoder_state_dict"], strict=True)
del baseline_payload

model_candidates = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        payload = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if payload.get("model_id") == "expansionnet_v2":
        model_candidates.append((manifest_path, payload))
assert len(model_candidates) == 1, model_candidates
MODEL_MANIFEST_PATH, model_manifest = model_candidates[0]
MODEL_ROOT = MODEL_MANIFEST_PATH.parent
TEACHER_CHECKPOINT_PATH = MODEL_ROOT / model_manifest["checkpoint"]["relative_path"]
NATIVE_VOCAB_PATH = MODEL_ROOT / model_manifest["vocabulary"]["relative_path"]
assert sha256(TEACHER_CHECKPOINT_PATH) == pseudo_result["checkpoint_sha256"]
assert sha256(TEACHER_CHECKPOINT_PATH) == model_manifest["checkpoint"]["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    native_vocab = pickle.load(handle)
NATIVE_WORD2IDX = native_vocab["word2idx_dict"]
NATIVE_IDX2WORD = native_vocab["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX["SOS"])
NATIVE_EOS = int(NATIVE_WORD2IDX["EOS"])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
assert len(NATIVE_IDX2WORD) == 10000

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
sys.path.insert(0, str(SOURCE_ROOT))
from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
teacher = End_ExpansionNet_v2(
    swin_img_size=384, swin_patch_size=4, swin_in_chans=3,
    swin_embed_dim=192, swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48], swin_window_size=12,
    swin_mlp_ratio=4.0, swin_qkv_bias=True, swin_qk_scale=None,
    swin_drop_rate=0.0, swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0, swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False, swin_patch_norm=True, swin_use_checkpoint=False,
    final_swin_dim=1536, d_model=512, N_enc=3, N_dec=3,
    num_heads=8, ff=2048,
    num_exp_enc_list=[32, 64, 128, 256, 512], num_exp_dec=16,
    output_word2idx=NATIVE_WORD2IDX, output_idx2word=NATIVE_IDX2WORD,
    max_seq_len=74, drop_args=drop_args, rank=DEVICE,
)
teacher_payload = torch.load(
    TEACHER_CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
load_result = teacher.load_state_dict(
    teacher_payload["model_state_dict"], strict=True
)
assert not load_result.missing_keys and not load_result.unexpected_keys
del teacher_payload
gc.collect()
teacher = teacher.to(DEVICE).eval()
for parameter in teacher.parameters():
    parameter.requires_grad_(False)

# Mapping riêng cho Flickr30k. Không dùng mapping MS COCO.
target_token_to_id = vocab["token_to_id"]
special_mapping = {
    "PAD": 0, "SOS": 1, "EOS": 2, "UNK": 3,
    "<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3,
}
mapping_pairs = []
for source_id, raw_token in enumerate(NATIVE_IDX2WORD):
    token = str(raw_token)
    target_id = special_mapping.get(token)
    if target_id is None:
        target_id = target_token_to_id.get(token.lower())
    if target_id is not None:
        mapping_pairs.append((int(source_id), int(target_id)))
assert len(mapping_pairs) >= 1000
assert (NATIVE_PAD, 0) in mapping_pairs
assert (NATIVE_BOS, 1) in mapping_pairs
assert (NATIVE_EOS, 2) in mapping_pairs

MAPPED_SOURCE = torch.tensor(
    [source for source, _ in mapping_pairs],
    dtype=torch.long, device=DEVICE,
)
MAPPED_TARGET = torch.tensor(
    [target for _, target in mapping_pairs],
    dtype=torch.long, device=DEVICE,
)
mapped_target_count = len(set(MAPPED_TARGET.detach().cpu().tolist()))
mapping_contract = {
    "schema_version": "f30k_t1s1_dense_mapping_v1",
    "source_vocabulary_size": 10000,
    "target_vocabulary_size": 7456,
    "mapped_source_entries": len(mapping_pairs),
    "mapped_target_tokens": mapped_target_count,
    "target_vocabulary_coverage": mapped_target_count / 7456,
    "collision_policy": "scatter_add_then_renormalize",
    "source_vocabulary_sha256": sha256(NATIVE_VOCAB_PATH),
    "target_vocabulary_sha256": sha256(VOCAB_PATH),
    "test_set_status": "sealed",
    "test_set_used": False,
}
(WORK_ROOT / "DENSE_MAPPING_CONTRACT.json").write_text(
    json.dumps(mapping_contract, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

pseudo_df = pd.read_parquet(PSEUDO_PARQUET_PATH)
assert len(pseudo_df) == pseudo_df["image_id"].nunique() == 29000
assert set(pseudo_df["image_id"].astype(int)) == set(train_df["image_id"].astype(int))
PSEUDO_BY_ID = {
    int(row.image_id): (
        list(map(int, row.teacher_token_ids)),
        list(map(int, row.student_token_ids)),
    )
    for row in pseudo_df.itertuples(index=False)
}
assert all(len(source) == len(target) for source, target in PSEUDO_BY_ID.values())
TRUNCATED_TEACHER_IDS = sorted(
    image_id
    for image_id, (source, _) in PSEUDO_BY_ID.items()
    if source[-1] != NATIVE_EOS
)
assert len(TRUNCATED_TEACHER_IDS) == 2, TRUNCATED_TEACHER_IDS
assert all(
    source[-1] == NATIVE_EOS
    or (len(source) == 31 and NATIVE_EOS not in source)
    for source, _ in PSEUDO_BY_ID.values()
)
assert all(target[0] == 1 and target[-1] == 2 for _, target in PSEUDO_BY_ID.values())

class DenseMappedDataset(Dataset):
    def __init__(self, rows):
        self.rows, self.epoch = rows, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        image_id, captions = self.rows[index]
        human = captions[(self.epoch + image_id % len(captions)) % len(captions)]
        teacher_ids, student_pseudo = PSEUDO_BY_ID[int(image_id)]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as source:
            image = source.convert("RGB")
            top, left, height, width = transforms.RandomResizedCrop.get_params(
                image, scale=(0.8, 1.0), ratio=(0.75, 1.3333333333333333)
            )
            student_image = TF.resized_crop(
                image, top, left, height, width, [224, 224],
                interpolation=InterpolationMode.BICUBIC,
            )
            teacher_image = TF.resized_crop(
                image, top, left, height, width, [384, 384],
                interpolation=InterpolationMode.BILINEAR,
            )
            if torch.rand(()) < 0.5:
                student_image = TF.hflip(student_image)
                teacher_image = TF.hflip(teacher_image)
            student_image = TF.normalize(TF.to_tensor(student_image), MEAN, STD)
            teacher_image = TF.normalize(
                TF.to_tensor(teacher_image),
                [0.485, 0.456, 0.406], [0.229, 0.224, 0.225],
            )
        return (
            int(image_id), student_image, teacher_image,
            torch.tensor(human, dtype=torch.long),
            torch.tensor(teacher_ids, dtype=torch.long),
            torch.tensor(student_pseudo, dtype=torch.long),
        )

def dense_collate(rows):
    image_ids, student_images, teacher_images, human, teacher_ids, pseudo = zip(*rows)
    def pad(values, pad_id):
        maximum = max(len(value) for value in values)
        tokens = torch.full(
            (len(values), maximum), int(pad_id), dtype=torch.long
        )
        mask = torch.zeros(len(values), maximum, dtype=torch.bool)
        for index, value in enumerate(values):
            tokens[index, :len(value)] = value
            mask[index, :len(value)] = True
        return tokens, mask
    human_tokens, human_mask = pad(human, 0)
    teacher_tokens, teacher_mask = pad(teacher_ids, NATIVE_PAD)
    pseudo_tokens, pseudo_mask = pad(pseudo, 0)
    assert torch.equal(teacher_mask, pseudo_mask)
    # Hai chuỗi Teacher chạm beam_max_seq_len=31 và không sinh EOS. Student
    # giữ tối đa 29 lexical token rồi thêm EOS, nên vị trí KD cuối của hai
    # chuỗi này không còn tương ứng token-với-token và phải được loại khỏi KL.
    kd_mask = teacher_mask[:, 1:].clone()
    for row_index, value in enumerate(teacher_ids):
        if int(value[-1]) != NATIVE_EOS:
            assert len(value) == 31 and NATIVE_EOS not in value
            kd_mask[row_index, len(value) - 2] = False
    return (
        torch.tensor(image_ids), torch.stack(student_images),
        torch.stack(teacher_images), human_tokens, human_mask,
        teacher_tokens, pseudo_tokens, pseudo_mask, kd_mask,
    )

train_set = DenseMappedDataset(train_groups)
def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    return DataLoader(
        train_set, batch_size=1, shuffle=True,
        generator=torch.Generator().manual_seed(SEED + epoch),
        num_workers=2, pin_memory=True, persistent_workers=False,
        collate_fn=dense_collate,
    )

BASELINE_CIDER = float(baseline_result["best_validation_metrics"]["CIDEr_D"])
BASELINE_SHA256 = sha256(BASELINE_BEST_PATH)
PSEUDO_SHA256 = sha256(PSEUDO_PARQUET_PATH)
TEACHER_SHA256 = sha256(TEACHER_CHECKPOINT_PATH)
MAPPING_SHA256 = sha256(WORK_ROOT / "DENSE_MAPPING_CONTRACT.json")
print("Dense mapping gate     : PASSED")
print("Source / target vocab  : 10000 / 7456")
print("Mapped entries/targets :", len(mapping_pairs), "/", mapped_target_count)
print("Teacher EOS/truncated  :", 29000 - len(TRUNCATED_TEACHER_IDS),
      "/", len(TRUNCATED_TEACHER_IDS))
print("Physical/accum/effect. : 1 / 64 / 64")
print("Baseline CIDEr-D       :", BASELINE_CIDER)
print("Held-out Test          : SEALED / NOT READ")

Dense mapping gate     : PASSED
Source / target vocab  : 10000 / 7456
Mapped entries/targets : 6187 / 6187
Teacher EOS/truncated  : 28998 / 2
Physical/accum/effect. : 1 / 64 / 64
Baseline CIDEr-D       : 0.4761966714974793
Held-out Test          : SEALED / NOT READ


## Cell 4 — Beam-3 và official Validation metrics

In [4]:
TOKENS = vocab["tokens"]
SPECIAL = {0, 1, 2, 3}

def ids_to_caption(ids):
    words = []
    for x in ids:
        x = int(x)
        if x == 2:
            break
        if x not in SPECIAL:
            words.append(TOKENS[x])
    text = " ".join(words)
    for p in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + p, p)
    return text.strip()

@torch.inference_mode()
def beam_decode(memory, beam_size=3, alpha=0.7):
    beams = [([1], 0.0, False)]
    for _ in range(30):
        active = [b for b in beams if not b[2]]
        if not active:
            break
        seq = torch.tensor([b[0] for b in active], device=DEVICE)
        mem = memory.expand(len(active), -1, -1)
        logits = decoder(seq, mem)[:, -1].float()
        logits[:, 0] = -torch.inf
        logits[:, 1] = -torch.inf
        logp = logits.log_softmax(-1)
        candidates = [b for b in beams if b[2]]
        for i, (tokens, score, _) in enumerate(active):
            values, indices = logp[i].topk(beam_size)
            for value, index in zip(values.tolist(), indices.tolist()):
                new = tokens + [index]
                candidates.append((new, score + value, index == 2))
        beams = sorted(candidates, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha), reverse=True)[:beam_size]
    return max(beams, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha))[0]

@torch.inference_mode()
def validate():
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    encoder.eval()
    decoder.eval()
    hypotheses = {}
    ce_sum, token_count = 0.0, 0
    for image_ids, images, tokens, mask in val_loader:
        images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            memory = encoder(images)
            logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
        targets = tokens[:, 1:]
        valid = mask[:, 1:]
        losses = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),
                                 targets.reshape(-1), reduction="none").reshape_as(targets)
        ce_sum += float((losses * valid).sum())
        token_count += int(valid.sum())
        for j, image_id in enumerate(image_ids.tolist()):
            hypotheses[int(image_id)] = [ids_to_caption(beam_decode(memory[j:j+1]))]
    refs = {int(k): [{"caption": x} for x in v] for k, v in VAL_REFERENCES.items()}
    hyps = {int(k): [{"caption": v[0]}] for k, v in hypotheses.items()}
    tokenizer = PTBTokenizer()
    refs = tokenizer.tokenize(refs)
    hyps = tokenizer.tokenize(hyps)
    b, _ = Bleu(4).compute_score(refs, hyps)
    meteor = Meteor()
    try:
        m, _ = meteor.compute_score(refs, hyps)
    finally:
        close_method = getattr(meteor, "close", None)
        if callable(close_method):
            close_method()
    r, _ = Rouge().compute_score(refs, hyps)
    c, _ = Cider().compute_score(refs, hyps)
    return {"ce_loss": ce_sum / token_count, "BLEU_1": b[0], "BLEU_4": b[3],
            "METEOR": m, "ROUGE_L": r, "CIDEr_D": c}, hypotheses

## Cell 5 — Full Dense Mapped Logit-KD và exact resume

In [5]:
MAX_EPOCHS = int(protocol["maximum_epochs"])
PATIENCE = int(protocol["early_stopping"]["patience"])
MIN_EPOCHS = int(protocol["early_stopping"]["minimum_completed_epochs"])
MIN_DELTA = float(protocol["minimum_cider_improvement"])
TEMPERATURE = 4.0
ALPHA_HUMAN = 0.5
ALPHA_KD = 0.5
ACCUM = 64
steps_per_epoch = math.ceil(len(train_set) / ACCUM)
total_steps = MAX_EPOCHS * steps_per_epoch
warmup = max(1, round(0.05 * total_steps))

optimizer = torch.optim.AdamW([
    {"params": encoder.backbone.parameters(), "lr": 1e-4},
    {"params": list(encoder.projection.parameters()) +
               [encoder.visual_position_embedding] +
               list(encoder.output_norm.parameters()), "lr": 3e-4},
    {"params": decoder.parameters(), "lr": 3e-4},
], betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)

def lr_lambda(step):
    if step < warmup:
        return max(1e-8, step / warmup)
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)
LAST = WORK_ROOT / "last.pt"
BEST = WORK_ROOT / "best.pt"
METRICS_CSV = WORK_ROOT / "metrics.csv"
CAPTIONS = WORK_ROOT / "best_validation_captions.json"

def state_cpu(module):
    return {key: value.detach().cpu() for key, value in module.state_dict().items()}

def teacher_mapped_distribution(teacher_images, teacher_tokens):
    decoder_input = teacher_tokens[:, :-1]
    labels = teacher_tokens[:, 1:]
    num_pads = (decoder_input == NATIVE_PAD).sum(dim=1).tolist()
    with torch.no_grad():
        teacher_logits = teacher(
            enc_x=teacher_images, dec_x=decoder_input,
            enc_x_num_pads=[0] * teacher_images.shape[0],
            dec_x_num_pads=[int(value) for value in num_pads],
            apply_log_softmax=False, mode="forward",
        )
        assert teacher_logits.shape[-1] == 10000
        probabilities = torch.softmax(
            teacher_logits.float() / TEMPERATURE, dim=-1
        )
        selected = probabilities.index_select(-1, MAPPED_SOURCE)
        dense = torch.zeros(
            *selected.shape[:-1], 7456,
            dtype=selected.dtype, device=DEVICE,
        )
        index = MAPPED_TARGET.view(1, 1, -1).expand_as(selected)
        dense.scatter_add_(-1, index, selected)
        mass = dense.sum(dim=-1)
        assert torch.isfinite(mass).all() and bool((mass > 0).all())
        distribution = dense / mass.unsqueeze(-1)
        valid = labels.ne(NATIVE_PAD)
    return distribution.detach(), valid.detach(), mass.detach()

def save_last(progress, history, best_cider, stale):
    payload = {
        "schema_version": "f30k_dense_mapped_logit_kd_v1",
        "experiment_id": EXPERIMENT_ID, "seed": SEED,
        "encoder_state_dict": state_cpu(encoder),
        "decoder_state_dict": state_cpu(decoder),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "progress": progress, "history": history,
        "best_cider": best_cider, "stale_evaluations": stale,
        "baseline_checkpoint_sha256": BASELINE_SHA256,
        "teacher_checkpoint_sha256": TEACHER_SHA256,
        "pseudo_corpus_sha256": PSEUDO_SHA256,
        "mapping_contract_sha256": MAPPING_SHA256,
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all(),
        "test_set_status": "sealed", "test_set_used": False,
    }
    temporary = LAST.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, LAST)

def save_best(epoch, metrics):
    temporary = BEST.with_suffix(".tmp")
    torch.save({
        "schema_version": "f30k_dense_mapped_logit_kd_best_v1",
        "experiment_id": EXPERIMENT_ID, "seed": SEED, "epoch": epoch,
        "encoder_state_dict": state_cpu(encoder),
        "decoder_state_dict": state_cpu(decoder),
        "metrics": metrics,
        "baseline_checkpoint_sha256": BASELINE_SHA256,
        "teacher_checkpoint_sha256": TEACHER_SHA256,
        "pseudo_corpus_sha256": PSEUDO_SHA256,
        "mapping_contract_sha256": MAPPING_SHA256,
        "test_set_status": "sealed", "test_set_used": False,
    }, temporary)
    os.replace(temporary, BEST)

start_epoch = cursor = global_step = 0
human_sum = kd_sum = mapped_mass_sum = 0.0
human_count = kd_count = mapped_mass_count = 0
history, best_cider, stale = [], -math.inf, 0
resume_candidates = sorted(INPUT_ROOT.rglob(f"{EXPERIMENT_ID}/last.pt"))
if resume_candidates:
    assert len(resume_candidates) == 1, resume_candidates
    payload = torch.load(resume_candidates[0], map_location="cpu", weights_only=False)
    assert payload["experiment_id"] == EXPERIMENT_ID
    assert payload["baseline_checkpoint_sha256"] == BASELINE_SHA256
    assert payload["teacher_checkpoint_sha256"] == TEACHER_SHA256
    assert payload["pseudo_corpus_sha256"] == PSEUDO_SHA256
    assert payload["mapping_contract_sha256"] == MAPPING_SHA256
    assert payload["test_set_used"] is False
    encoder.load_state_dict(payload["encoder_state_dict"], strict=True)
    decoder.load_state_dict(payload["decoder_state_dict"], strict=True)
    optimizer.load_state_dict(payload["optimizer_state_dict"])
    scheduler.load_state_dict(payload["scheduler_state_dict"])
    scaler.load_state_dict(payload["scaler_state_dict"])
    random.setstate(payload["python_rng_state"])
    np.random.set_state(payload["numpy_rng_state"])
    torch.set_rng_state(payload["torch_rng_state"])
    torch.cuda.set_rng_state_all(payload["cuda_rng_state_all"])
    progress = payload["progress"]
    start_epoch = int(progress["epoch"])
    cursor = int(progress["micro_cursor"])
    global_step = int(progress["global_step"])
    human_sum = float(progress["human_sum"])
    human_count = int(progress["human_count"])
    kd_sum = float(progress["kd_sum"])
    kd_count = int(progress["kd_count"])
    mapped_mass_sum = float(progress["mapped_mass_sum"])
    mapped_mass_count = int(progress["mapped_mass_count"])
    history = payload["history"]
    best_cider = float(payload["best_cider"])
    stale = int(payload["stale_evaluations"])
    resume_root = resume_candidates[0].parent
    shutil.copy2(resume_candidates[0], LAST)
    for name, destination in [
        ("best.pt", BEST), ("metrics.csv", METRICS_CSV),
        ("best_validation_captions.json", CAPTIONS),
    ]:
        source = resume_root / name
        if source.is_file():
            shutil.copy2(source, destination)
    print("Exact resume           :", resume_candidates[0])
    print("Epoch / cursor / step  :", start_epoch + 1, cursor, global_step)

run_started = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    loader = make_train_loader(epoch)
    encoder.train(); decoder.train(); teacher.eval()
    for module in encoder.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
    current_cursor = cursor if epoch == start_epoch else 0
    if current_cursor == 0:
        human_sum = kd_sum = mapped_mass_sum = 0.0
        human_count = kd_count = mapped_mass_count = 0
    optimizer.zero_grad(set_to_none=True)
    accumulation_count = 0
    for micro_index, batch in enumerate(loader):
        if micro_index < current_cursor:
            continue
        (
            _, student_images, teacher_images, human, human_mask,
            teacher_tokens, pseudo, pseudo_mask, kd_mask,
        ) = batch
        student_images = student_images.to(DEVICE, non_blocking=True)
        teacher_images = teacher_images.to(DEVICE, non_blocking=True)
        human, human_mask = human.to(DEVICE), human_mask.to(DEVICE)
        teacher_tokens = teacher_tokens.to(DEVICE)
        pseudo, pseudo_mask = pseudo.to(DEVICE), pseudo_mask.to(DEVICE)
        kd_mask = kd_mask.to(DEVICE)
        q_teacher, teacher_valid, mapped_mass = teacher_mapped_distribution(
            teacher_images, teacher_tokens
        )
        assert torch.equal(teacher_valid, pseudo_mask[:, 1:])
        assert torch.all(kd_mask <= teacher_valid)
        kd_valid = teacher_valid & kd_mask
        assert bool(kd_valid.any())
        with torch.autocast("cuda", dtype=torch.float16):
            memory = encoder(student_images)
            human_logits = decoder(human[:, :-1], memory, human_mask[:, :-1]).float()
            pseudo_logits = decoder(pseudo[:, :-1], memory, pseudo_mask[:, :-1]).float()
            human_targets, human_valid = human[:, 1:], human_mask[:, 1:]
            human_raw = F.cross_entropy(
                human_logits.reshape(-1, 7456), human_targets.reshape(-1),
                reduction="none",
            ).reshape_as(human_targets)
            human_ce = (human_raw * human_valid).sum() / human_valid.sum()
            student_log_prob = F.log_softmax(
                pseudo_logits / TEMPERATURE, dim=-1
            )
            kd_raw = F.kl_div(
                student_log_prob, q_teacher, reduction="none"
            ).sum(dim=-1)
            kd = (kd_raw * kd_valid).sum() / kd_valid.sum()
            loss = (
                ALPHA_HUMAN * human_ce +
                ALPHA_KD * (TEMPERATURE ** 2) * kd
            )
        scaler.scale(loss / ACCUM).backward()
        accumulation_count += 1
        human_sum += float((human_raw.detach() * human_valid).sum())
        human_count += int(human_valid.sum())
        kd_sum += float((kd_raw.detach() * kd_valid).sum())
        kd_count += int(kd_valid.sum())
        mapped_mass_sum += float(mapped_mass[kd_valid].sum())
        mapped_mass_count += int(kd_valid.sum())
        final_micro = micro_index + 1 == len(loader)
        if accumulation_count == ACCUM or final_micro:
            scaler.unscale_(optimizer)
            if final_micro and accumulation_count < ACCUM:
                correction = ACCUM / accumulation_count
                for parameter in list(encoder.parameters()) + list(decoder.parameters()):
                    if parameter.grad is not None:
                        parameter.grad.mul_(correction)
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(decoder.parameters()), 1.0
            )
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1
            accumulation_count = 0
            if global_step % 100 == 0 and not final_micro:
                save_last({
                    "epoch": epoch, "micro_cursor": micro_index + 1,
                    "global_step": global_step,
                    "human_sum": human_sum, "human_count": human_count,
                    "kd_sum": kd_sum, "kd_count": kd_count,
                    "mapped_mass_sum": mapped_mass_sum,
                    "mapped_mass_count": mapped_mass_count,
                }, history, best_cider, stale)
        if (micro_index + 1) % 3200 == 0 or final_micro:
            print(
                f"epoch={epoch+1:02d} images={micro_index+1:05d}/{len(loader):05d} "
                f"step={global_step:05d} CE={human_sum/human_count:.6f} "
                f"KL={kd_sum/kd_count:.6f} "
                f"mass={mapped_mass_sum/mapped_mass_count:.6f}",
                flush=True,
            )
    print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | VALIDATION START", flush=True)
    metrics, hypotheses = validate()
    metrics.update({
        "epoch": epoch + 1,
        "train_human_ce": human_sum / human_count,
        "train_mapped_kl": kd_sum / kd_count,
        "train_weighted_loss": (
            ALPHA_HUMAN * human_sum / human_count +
            ALPHA_KD * (TEMPERATURE ** 2) * kd_sum / kd_count
        ),
        "mapped_mass_mean": mapped_mass_sum / mapped_mass_count,
        "global_step": global_step,
        "elapsed_hours": (time.time() - run_started) / 3600,
    })
    history.append(metrics)
    pd.DataFrame(history).to_csv(METRICS_CSV, index=False)
    improved = metrics["CIDEr_D"] > best_cider + MIN_DELTA
    if improved:
        best_cider, stale = metrics["CIDEr_D"], 0
    else:
        stale += 1
    if improved:
        save_best(epoch, metrics)
        CAPTIONS.write_text(
            json.dumps(
                {"references": VAL_REFERENCES, "hypotheses": hypotheses},
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
    save_last({
        "epoch": epoch + 1, "micro_cursor": 0, "global_step": global_step,
        "human_sum": 0.0, "human_count": 0,
        "kd_sum": 0.0, "kd_count": 0,
        "mapped_mass_sum": 0.0, "mapped_mass_count": 0,
    }, history, best_cider, stale)
    print(
        f"Epoch {epoch+1:02d} | CIDEr-D={metrics['CIDEr_D']:.6f} | "
        f"delta_baseline={metrics['CIDEr_D']-BASELINE_CIDER:+.6f} | "
        f"stale={stale}/{PATIENCE}",
        flush=True,
    )
    cursor = 0
    if epoch + 1 >= MIN_EPOCHS and stale >= PATIENCE:
        print("Early stopping         : TRIGGERED")
        break

assert BEST.exists() and LAST.exists()
best_payload = torch.load(BEST, map_location="cpu", weights_only=False)
best_metrics = best_payload["metrics"]
summary = {
    "schema_version": "f30k_dense_mapped_logit_kd_result_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete", "experiment_id": EXPERIMENT_ID,
    "teacher_id": "T1", "teacher_name": TEACHER_NAME,
    "student_id": "S1", "student_name": MODEL_NAME, "seed": SEED,
    "method": "Dense Mapped Logit-KD",
    "temperature": TEMPERATURE,
    "loss": "0.5 CE_human + 0.5 T^2 KL_mapped",
    "best_epoch": int(best_payload["epoch"]) + 1,
    "best_validation_metrics": best_metrics,
    "baseline_validation_cider_d": BASELINE_CIDER,
    "delta_validation_cider_d": float(best_metrics["CIDEr_D"]) - BASELINE_CIDER,
    "mapping_contract": mapping_contract,
    "teacher_sequences_with_eos": 29000 - len(TRUNCATED_TEACHER_IDS),
    "teacher_sequences_truncated_at_max_length": len(TRUNCATED_TEACHER_IDS),
    "truncated_final_kd_positions_masked": len(TRUNCATED_TEACHER_IDS),
    "selection_metric": "Validation CIDEr-D",
    "maximum_epochs": MAX_EPOCHS, "early_stopping_enabled": True,
    "next_phase_authorized": "F30K_T1S1_DenseMap_Aggregate",
    "test_set_status": "sealed", "test_set_used": False,
    "artifacts": {},
}
for path in [
    BEST, LAST, METRICS_CSV, CAPTIONS,
    WORK_ROOT / "DENSE_MAPPING_CONTRACT.json",
]:
    summary["artifacts"][path.name] = {
        "bytes": path.stat().st_size, "sha256": sha256(path)
    }
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
result_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
manifest_path = WORK_ROOT / "MANIFEST.json"
artifacts = []
for path in [result_path] + [
    BEST, LAST, METRICS_CSV, CAPTIONS,
    WORK_ROOT / "DENSE_MAPPING_CONTRACT.json",
]:
    artifacts.append({
        "relative_path": path.name, "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
manifest_path.write_text(json.dumps({
    "schema_version": "f30k_dense_mapped_manifest_v1",
    "experiment_id": EXPERIMENT_ID, "status": "complete",
    "artifacts": artifacts,
    "test_set_status": "sealed", "test_set_used": False,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Status                 : COMPLETE")
print("Best Validation        :", best_metrics)
print("Delta vs baseline      :", summary["delta_validation_cider_d"])
print("Next                   :", summary["next_phase_authorized"])
print("Held-out Test          : SEALED / NOT USED")

Exact resume           : /kaggle/input/notebooks/lychiity/f30k-t1s1-densemap-123/F30K_T1S1_DenseMap_123/last.pt
Epoch / cursor / step  : 11 10240 4700
epoch=11 images=12800/29000 step=04740 CE=3.271626 KL=0.163546 mass=0.668901
epoch=11 images=16000/29000 step=04790 CE=3.273454 KL=0.163692 mass=0.668828
epoch=11 images=19200/29000 step=04840 CE=3.272066 KL=0.163729 mass=0.668791
epoch=11 images=22400/29000 step=04890 CE=3.272600 KL=0.163988 mass=0.668758
epoch=11 images=25600/29000 step=04940 CE=3.275503 KL=0.163812 mass=0.668718
epoch=11 images=28800/29000 step=04990 CE=3.277673 KL=0.163788 mass=0.668664
epoch=11 images=29000/29000 step=04994 CE=3.277518 KL=0.163754 mass=0.668667
Epoch 11/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 345698.57 tokens per second.
PTBTokenizer tokenized 10407 tokens at 114685.04 tokens per second.


{'testlen': 9367, 'reflen': 9582, 'guess': [9367, 8353, 7339, 6325], 'correct': [6038, 2556, 905, 314]}
ratio: 0.977562095595807
Epoch 11 | CIDEr-D=0.358060 | delta_baseline=-0.118137 | stale=1/4
epoch=12 images=03200/29000 step=05044 CE=3.280876 KL=0.158413 mass=0.667998
epoch=12 images=06400/29000 step=05094 CE=3.271817 KL=0.157842 mass=0.668280
epoch=12 images=09600/29000 step=05144 CE=3.258475 KL=0.158117 mass=0.668605
epoch=12 images=12800/29000 step=05194 CE=3.262844 KL=0.158540 mass=0.668586
epoch=12 images=16000/29000 step=05244 CE=3.272394 KL=0.158495 mass=0.668603
epoch=12 images=19200/29000 step=05294 CE=3.265943 KL=0.158421 mass=0.668662
epoch=12 images=22400/29000 step=05344 CE=3.269012 KL=0.158748 mass=0.668616
epoch=12 images=25600/29000 step=05394 CE=3.268888 KL=0.158732 mass=0.668659
epoch=12 images=28800/29000 step=05444 CE=3.269450 KL=0.158652 mass=0.668670
epoch=12 images=29000/29000 step=05448 CE=3.270007 KL=0.158695 mass=0.668665
Epoch 12/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 386363.49 tokens per second.
PTBTokenizer tokenized 10552 tokens at 105721.93 tokens per second.


{'testlen': 9503, 'reflen': 9641, 'guess': [9503, 8489, 7475, 6461], 'correct': [6138, 2628, 940, 327]}
ratio: 0.9856861321438661
Epoch 12 | CIDEr-D=0.363538 | delta_baseline=-0.112659 | stale=2/4
epoch=13 images=03200/29000 step=05498 CE=3.207847 KL=0.154658 mass=0.668405
epoch=13 images=06400/29000 step=05548 CE=3.226754 KL=0.155989 mass=0.668694
epoch=13 images=09600/29000 step=05598 CE=3.244380 KL=0.155637 mass=0.668813
epoch=13 images=12800/29000 step=05648 CE=3.241365 KL=0.155439 mass=0.668783
epoch=13 images=16000/29000 step=05698 CE=3.238059 KL=0.155354 mass=0.668632
epoch=13 images=19200/29000 step=05748 CE=3.236222 KL=0.155206 mass=0.668628
epoch=13 images=22400/29000 step=05798 CE=3.236312 KL=0.155281 mass=0.668629
epoch=13 images=25600/29000 step=05848 CE=3.234371 KL=0.155084 mass=0.668692
epoch=13 images=28800/29000 step=05898 CE=3.234632 KL=0.155106 mass=0.668710
epoch=13 images=29000/29000 step=05902 CE=3.235284 KL=0.155093 mass=0.668704
Epoch 13/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 358321.21 tokens per second.
PTBTokenizer tokenized 10541 tokens at 97980.53 tokens per second.


... (7 outputs left out)

PTBTokenizer tokenized 74030 tokens at 336234.96 tokens per second.
PTBTokenizer tokenized 10525 tokens at 107517.24 tokens per second.


{'testlen': 9490, 'reflen': 9637, 'guess': [9490, 8476, 7462, 6448], 'correct': [6126, 2615, 927, 328]}
ratio: 0.984746290339215
Epoch 17 | CIDEr-D=0.366866 | delta_baseline=-0.109331 | stale=3/4
epoch=18 images=03200/29000 step=07768 CE=3.200236 KL=0.142134 mass=0.668466
epoch=18 images=06400/29000 step=07818 CE=3.182730 KL=0.142369 mass=0.668782
epoch=18 images=09600/29000 step=07868 CE=3.170501 KL=0.142639 mass=0.668816
epoch=18 images=12800/29000 step=07918 CE=3.162568 KL=0.142500 mass=0.668805
epoch=18 images=16000/29000 step=07968 CE=3.161743 KL=0.142591 mass=0.668793
epoch=18 images=19200/29000 step=08018 CE=3.162942 KL=0.142487 mass=0.668791
epoch=18 images=22400/29000 step=08068 CE=3.161749 KL=0.142491 mass=0.668764
epoch=18 images=25600/29000 step=08118 CE=3.160070 KL=0.142526 mass=0.668703
epoch=18 images=28800/29000 step=08168 CE=3.161530 KL=0.142735 mass=0.668683
epoch=18 images=29000/29000 step=08172 CE=3.161034 KL=0.142737 mass=0.668690
Epoch 18/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 397177.30 tokens per second.
PTBTokenizer tokenized 10620 tokens at 97630.26 tokens per second.


{'testlen': 9581, 'reflen': 9669, 'guess': [9581, 8567, 7553, 6539], 'correct': [6196, 2660, 942, 328]}
ratio: 0.9908987485778269
Epoch 18 | CIDEr-D=0.371922 | delta_baseline=-0.104275 | stale=4/4
Early stopping         : TRIGGERED
Status                 : COMPLETE
Best Validation        : {'ce_loss': 3.5384845291444287, 'BLEU_1': 0.635915292312738, 'BLEU_4': 0.18389580992219096, 'METEOR': 0.1742591996478866, 'ROUGE_L': np.float64(0.43177867686958016), 'CIDEr_D': np.float64(0.37218297186251), 'epoch': 14, 'train_human_ce': 3.224121340143779, 'train_mapped_kl': 0.15159110710204188, 'train_weighted_loss': 2.8247895268882246, 'mapped_mass_mean': 0.6686782978188223, 'global_step': 6356, 'elapsed_hours': 4.42724278555976}
Delta vs baseline      : -0.10401369963496931
Next                   : F30K_T1S1_DenseMap_Aggregate
Held-out Test          : SEALED / NOT USED
